# Check de versión de datos y splits

Este notebook comprueba si `X_train/X_test/y_train/y_test` son coherentes con `df_PCA_95.csv` y `df_victima_target.csv`, usando el mismo `train_test_split(random_state=42, stratify=target)` del notebook original.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Cambia las rutas si tus ficheros están en otra carpeta
df_pca = pd.read_csv("df_PCA_95.csv", index_col=0)
y = pd.read_csv("df_victima_target.csv", index_col=0).squeeze()

X_train = pd.read_csv("X_train.csv", index_col=0)
X_test  = pd.read_csv("X_test.csv", index_col=0)
y_train = pd.read_csv("y_train.csv", index_col=0).squeeze()
y_test  = pd.read_csv("y_test.csv", index_col=0).squeeze()

print("df_pca:", df_pca.shape)
print("target:", y.shape)
print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)
print("Total splits:", len(X_train) + len(X_test))

In [ ]:
# Regeneramos los índices tal como hace predictor_victim.ipynb
idx_train, idx_test = train_test_split(
    df_pca.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Mismos índices train:", set(idx_train) == set(X_train.index))
print("Mismos índices test:", set(idx_test) == set(X_test.index))
print("Train y test no se solapan:", len(set(X_train.index) & set(X_test.index)) == 0)
print("Cubren todo df_pca:", set(X_train.index) | set(X_test.index) == set(df_pca.index))

print("y_train coincide con target:", y_train.equals(y.loc[X_train.index]))
print("y_test coincide con target:", y_test.equals(y.loc[X_test.index]))

In [ ]:
# Comparamos valores PCA.
# Nota: en PCA el signo de una componente puede cambiar si se recalcula.
# Por eso miramos coincidencia exacta y también si alguna PC está invertida de signo.

for split_name, X in [("train", X_train), ("test", X_test)]:
    print("\n---", split_name, "---")
    for col in df_pca.columns:
        a = df_pca.loc[X.index, col]
        b = X[col]
        same = np.allclose(a, b)
        opposite = np.allclose(a, -b)
        if not same:
            print(col, "same:", same, "opposite_sign:", opposite, "corr:", round(np.corrcoef(a, b)[0,1], 6))

## Interpretación rápida

- Si `Mismos índices train/test = True`, los splits proceden del mismo split reproducible.
- Si `y_train/y_test coincide = True`, las etiquetas coinciden.
- Si alguna PC aparece como `opposite_sign=True`, no implica necesariamente un error: en PCA el signo del eje puede invertirse al recalcular. Pero indica que no es exactamente el mismo CSV byte a byte.